In [ ]:
!pip install ultralytics timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/vinbig/vinbig_images.zip" -d /content/
!unzip -q "/content/drive/MyDrive/vinbig/vinbig_labels.zip" -d /content/

In [ ]:
import os

os.listdir("/content")

['.config', 'kaggle', 'drive', 'sample_data']

In [ ]:
!apt-get install tree

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 45 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (159 kB/s)
Selecting previously unselected package tree.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!tree -L 3 /content

/content
├── drive
│   └── MyDrive
│       ├── 1M1B PROJECT.pdf
│       ├── 3144301 - UNIT - 1_copy.pdf
│       ├── ASF's OMAD guide.docx
│       ├── Certificate (5)_compressed (1).pdf
│       ├── Certificate (5)_compressed.pdf
│       ├── Classroom
│       ├── Colab Notebooks
│       ├── Completion Certificate _ 1M1B.pdf
│       ├── Completion Certificate _ SkillsBuild.pdf
│       ├── Final Project ppt template (1).pptx
│       ├── Final Project ppt template.pptx
│       ├── Flipkart PYQ - Arsh Goyal.gdoc
│       ├── Frontend Developer Task.docx
│       ├── INDEX WAD.pdf
│       ├── Kalpana – Coding Jr Internship Assignment
│       ├── kp final resume.pdf
│       ├── Notes Sem 3 
│       ├── PPT  CYBER GYAN VIRTUAL INTERNSHIP.pptx
│       ├── PROJECT REPORT CYBER GYAN VIRTUAL INTERNSHIP.docx
│       ├── Screenshot 2026-01-07 150233.png
│       ├── Screenshot 2026-01-07 150406.png
│       ├── Screenshot 2026-01-07 150619.png
│       ├── Sem 6 paper
│       ├── Tech Intern L2 Assignment

In [ ]:
!find /content -type d -name "train"

/content/kaggle/working/vinbig_yolo/labels/train
/content/kaggle/working/vinbig_yolo/images/train


In [ ]:
img_dir = "/content/kaggle/working/vinbig_yolo/images/train"
label_dir = "/content/kaggle/working/vinbig_yolo/labels/train"

In [ ]:
img_dir = "/content/kaggle/working/vinbig_yolo/images/train"
label_dir = "/content/kaggle/working/vinbig_yolo/labels/train"

import os
print("Images:", len(os.listdir(img_dir)))
print("Labels:", len(os.listdir(label_dir)))

Images: 15000
Labels: 4394


PREPARE YOLO DATASET

In [ ]:
import shutil

train_img = "/content/yolo/images/train"
train_lbl = "/content/yolo/labels/train"

os.makedirs(train_img, exist_ok=True)
os.makedirs(train_lbl, exist_ok=True)

for lbl in os.listdir(label_dir):
    img = lbl.replace(".txt",".jpg")

    shutil.copy(f"{img_dir}/{img}", f"{train_img}/{img}")
    shutil.copy(f"{label_dir}/{lbl}", f"{train_lbl}/{lbl}")

print("YOLO Images:", len(os.listdir(train_img)))

YOLO Images: 4394


CREATE YOLO CONFIG

In [ ]:
data_yaml = """
path: /content/yolo
train: images/train
val: images/train

names:
  0: Aortic enlargement
  1: Atelectasis
  2: Calcification
  3: Cardiomegaly
  4: Consolidation
  5: ILD
  6: Infiltration
  7: Lung Opacity
  8: Nodule/Mass
  9: Other lesion
  10: Pleural effusion
  11: Pleural thickening
  12: Pneumothorax
  13: Pulmonary fibrosis
"""

with open("/content/yolo/data.yaml","w") as f:
    f.write(data_yaml)

In [ ]:
import os

label_path = "/content/yolo/labels/train"

for file in os.listdir(label_path):

    file_path = os.path.join(label_path, file)

    new_lines = []

    with open(file_path, "r") as f:
        lines = f.readlines()

        for line in lines:
            parts = list(map(float, line.strip().split()))

            cls = int(parts[0])
            x, y, w, h = parts[1:]

            # CLAMP values between 0 and 1
            x = min(max(x, 0), 1)
            y = min(max(y, 0), 1)
            w = min(max(w, 0), 1)
            h = min(max(h, 0), 1)

            new_lines.append(f"{cls} {x} {y} {w} {h}\n")

    with open(file_path, "w") as f:
        f.writelines(new_lines)

print("Labels fixed ✅")

Labels fixed ✅


TRAIN YOLO MODEL

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/yolo/data.yaml",
    epochs=5,
    imgsz=512,
    batch=16,
    cache=True
)

Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, p

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x783e6b6c6b40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.0

CELL 8 — Evaluate YOLO

In [ ]:
metrics = model.val()

print("YOLO Results")
print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)

Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,008,378 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 132.4±155.0 MB/s, size: 123.4 KB)
val: Scanning /content/yolo/labels/train.cache... 4394 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4394/4394 970.0Mit/s 0.0s
train: /content/yolo/images/train/01c2b9fcb0384c84648ed76c736552a8.jpg: 1 duplicate labels removed
train: /content/yolo/images/train/15acee7728e6530dfa2bd01521c7148d.jpg: 1 duplicate labels removed
train: /content/yolo/images/train/305c85dddfdddac905f288a8106ca371.jpg: 1 duplicate labels removed
train: /content/yolo/images/train/57a11dff1b33a1e011eb9ccd9a2f24cc.jpg: 1 duplicate labels removed
train: /content/yolo/images/train/58ab031cf2346b3a7b8fb32fb9ccd1c1.jpg: 1 duplicate labels removed
train: /content/yolo/images/train/613fd7f3a8b48248d4027c90a414ff9a.jpg: 1 duplicate labels removed
train: /content/yolo/image